# OW v3 mask quality audit
Train/Validation metadata and overlays only. No training or Test quality analysis. Attach Wild and OW00 v7. CPU is sufficient.


In [ ]:
import importlib
import subprocess
import sys

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

missing = []
for mod, pip_name in {
    "cv2": "opencv-python-headless",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "tqdm": "tqdm",
}.items():
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(pip_name)
if missing:
    _pip_install(missing)

import time
import torch
NOTEBOOK_START_TIME = time.time()
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import shutil
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision.models import EfficientNet_B7_Weights, efficientnet_b7
from tqdm.auto import tqdm


In [ ]:
def discover_wild_root() -> Path:
    roots = []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    roots.extend([Path.cwd() / "openforensics-wild-dataset", Path.cwd() / "Dataset", Path.cwd()])
    for root in roots:
        if not root.exists():
            continue
        for dirpath, dirnames, _ in os.walk(root):
            p = Path(dirpath)
            if "ow_masks" in p.parts:
                continue
            kids = {d.lower() for d in dirnames}
            if {"train", "validation", "test"} <= kids or {"train", "val", "test"} <= kids:
                return p
            if p.name.lower() == "dataset" and ("train" in kids or "test" in kids):
                return p
    raise FileNotFoundError("Attach ahmedgamal/openforensics-wild-dataset.")


def _split_dir(root: Path, name: str) -> Path:
    aliases = {
        "train": ["Train", "train"],
        "val": ["Validation", "Val", "validation", "val"],
        "test": ["Test", "test"],
    }
    for a in aliases[name]:
        p = root / a
        if p.is_dir():
            return p
    raise FileNotFoundError(f"No {name} split under {root}")


WILD_ROOT = discover_wild_root()
IN_KAGGLE = Path("/kaggle/working").exists()
WORKING = Path("/kaggle/working") if IN_KAGGLE else Path.cwd() / "kaggle_working"
WORKING.mkdir(parents=True, exist_ok=True)
for sub in ("checkpoints", "predictions", "logs"):
    (WORKING / sub).mkdir(parents=True, exist_ok=True)
print("wild root:", WILD_ROOT)
print("working:", WORKING)


In [ ]:
EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
LABELS = {"real": 0, "fake": 1}


def rel_key(split: str, class_name: str, file_name: str) -> str:
    return f"{split}/{class_name}/{file_name}"


def _class_dir(split_dir: Path, name: str):
    for a in (name, name.capitalize(), name.upper(), name.lower()):
        p = split_dir / a
        if p.is_dir():
            return p
    return None


def list_images(split_name: str) -> pd.DataFrame:
    split_dir = _split_dir(WILD_ROOT, split_name)
    rows = []
    for cname, lab in LABELS.items():
        cdir = _class_dir(split_dir, cname)
        if cdir is None:
            print("WARNING: missing", split_name, cname)
            continue
        for p in sorted(cdir.iterdir()):
            if p.is_file() and p.suffix.lower() in EXTS:
                rows.append({
                    "path": str(p),
                    "split": split_name,
                    "label": lab,
                    "class_name": cname,
                    "file_name": p.name,
                    "rel_key": rel_key(split_name, cname, p.name),
                })
    return pd.DataFrame(rows)


def unpack_mask_outputs(input_root: Path) -> list:
    import tempfile
    import zipfile
    from pathlib import PurePosixPath

    roots = []
    for archive in input_root.rglob("_output_.zip"):
        with zipfile.ZipFile(archive) as z:
            members = [m for m in z.infolist() if not m.is_dir()
                       and "ow_masks" in PurePosixPath(m.filename.replace("\\", "/")).parts]
            if not members:
                continue
            target = Path(tempfile.mkdtemp(prefix="veritas_ow_masks_"))
            for member in members:
                rel = PurePosixPath(member.filename.replace("\\", "/"))
                if rel.is_absolute() or ".." in rel.parts:
                    raise ValueError(f"Unsafe mask archive path: {member.filename}")
                rel = Path(*rel.parts[rel.parts.index("ow_masks"):])
                dst = target / rel
                if not dst.resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe mask archive path: {member.filename}")
                dst.parent.mkdir(parents=True, exist_ok=True)
                with z.open(member) as src, dst.open("wb") as out:
                    shutil.copyfileobj(src, out)
            roots.append(target)
            print("unpacked mask output", archive, "files", len(members), "into", target)
    return roots


def load_mask_table() -> pd.DataFrame:
    frames = []
    roots = []
    for p in (Path("/kaggle/input"), Path("/kaggle/working"), WORKING, Path.cwd()):
        if p.exists():
            roots.append(p)
    roots.extend(unpack_mask_outputs(Path("/kaggle/input")))
    seen = set()
    for root in roots:
        for dirpath, _, filenames in os.walk(root):
            if "index.csv" not in filenames:
                continue
            path = Path(dirpath) / "index.csv"
            key = str(path)
            if key in seen:
                continue
            seen.add(key)
            try:
                df = pd.read_csv(path, dtype=str)
            except Exception as e:
                print("skip", path, e)
                continue
            if not {"image_id", "rel_key", "mask_file", "status", "crop_x0"} <= set(df.columns):
                continue
            df["_root"] = str(path.parent)
            frames.append(df)
            print("mask index", path, "rows", len(df))
    if not frames:
        raise FileNotFoundError(
            "No ow_masks/index.csv found. Run OW_00_generate_masks and attach that notebook output."
        )
    all_df = pd.concat(frames, ignore_index=True)
    kept = []
    used = set()
    for rec in all_df.to_dict("records"):
        iid = str(rec["image_id"])
        if iid in used or str(rec.get("status")) != "ok":
            continue
        png = Path(str(rec["_root"])) / str(rec["mask_file"])
        if not png.is_file():
            continue
        rec["mask_path"] = str(png)
        kept.append(rec)
        used.add(iid)
    if not kept:
        raise FileNotFoundError("index.csv rows were found, but none of the PNG files are present.")
    out = pd.DataFrame(kept).drop_duplicates("rel_key", keep="last")
    print("masks ready", len(out), "teachers", sorted(set(out.teacher.dropna().astype(str))) if "teacher" in out.columns else [])
    return out


MASKS = load_mask_table()
assert len(MASKS) == 190335


In [ ]:
def letterbox_rgb(image_rgb, size: int):
    h, w = image_rgb.shape[:2]
    scale = min(size / max(h, 1), size / max(w, 1))
    nw, nh = max(int(round(w * scale)), 1), max(int(round(h * scale)), 1)
    resized = cv2.resize(image_rgb, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.zeros((size, size, 3), dtype=np.uint8)
    left, top = (size - nw) // 2, (size - nh) // 2
    canvas[top: top + nh, left: left + nw] = resized
    return canvas


In [ ]:
import matplotlib.pyplot as plt

summary, selected = [], []
for split in ("train", "val"):
    frame = list_images(split).merge(MASKS[["rel_key", "mask_path", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "positive_frac"]], on="rel_key", how="left")
    if frame.mask_path.isna().any():
        raise RuntimeError(f"Missing masks in {split}")
    frame["positive_frac"] = pd.to_numeric(frame.positive_frac, errors="raise")
    for label, group in frame.groupby("label"):
        summary.append(dict(split=split, label=int(label), n=len(group),
            teacher_positive_mean=float(group.positive_frac.mean()),
            teacher_empty_rate=float((group.positive_frac == 0).mean()),
            teacher_positive_std=float(group.positive_frac.std())))
        selected.append(group.sample(n=min(8, len(group)), random_state=42))
audit = pd.DataFrame(summary)
audit.to_csv(WORKING / "mask_quality_train_val.csv", index=False)
print(audit.to_string(index=False))
print("Real training targets become ZERO; nonempty fake masks remain uncertain teacher targets.")
print("Area is not confidence. No independent localization accuracy is established.")
samples = pd.concat(selected, ignore_index=True)
fig, axes = plt.subplots(8, 4, figsize=(16, 28))
for ax, row in zip(axes.flat, samples.itertuples()):
    bgr = cv2.imread(row.path)
    mask = cv2.imread(row.mask_path, cv2.IMREAD_GRAYSCALE)
    if bgr is None or mask is None or mask.shape != (600, 600):
        raise RuntimeError("Unreadable image or invalid saved mask")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    crop = rgb[int(float(row.crop_y0)):int(float(row.crop_y1)), int(float(row.crop_x0)):int(float(row.crop_x1))]
    canvas = letterbox_rgb(crop, 600)
    ax.imshow(canvas)
    ax.imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.35, vmin=0, vmax=255)
    ax.set_title(f"{row.split} {'Real' if row.label == 0 else 'Fake'} | teacher {row.positive_frac:.3f}")
    ax.axis("off")
for ax in axes.flat[len(samples):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(WORKING / "mask_audit_train_val.png", dpi=100)
plt.show()
print("MASK_AUDIT_COMPLETED_TRAIN_VAL_ONLY", len(MASKS))
